In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Priya Demographics works starts here 

In [ ]:
# Hackathon Data Cleaning Pipeline: Demographic Dataset

## Executive Overview
This notebook performs data cleaning, outlier handling, type casting, and feature engineering 
on the demographic patient dataset. 

### Data Quality Issues Addressed:
- Removal of corrupted baseline test records.
- Proper data type conversion for ID strings and categorical variables.
- Detection and correction of biologically impossible heights (< 1.0) 
- and weights (< 10.0 kg).
- Recalculation of Body Mass Index (BMI).
- Standardizing messy text entries and feature engineering clinical risk categories.


In [13]:
import numpy as np
import pandas as pd

In [22]:
# 1. Load raw dataset
df_demography = pd.read_csv("cardiac_failure/demography.csv")

In [23]:
# 2. Create working copy to preserve raw baseline
df_clean = df_demography.copy()

In [21]:
# 3. Baseline Audit
print(f"Dataset Dimensions: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
print("\n--- Initial Data Types ---")
print(df_clean.dtypes)
print("\n--- Initial Null Counts ---")
print(df_clean.isnull().sum())

Dataset Dimensions: 2009 rows, 7 columns

--- Initial Data Types ---
inpatient_number      int64
gender                  str
weight              float64
height              float64
bmi                 float64
occupation              str
agecat                  str
dtype: object

--- Initial Null Counts ---
inpatient_number     0
gender               1
weight               1
height               1
bmi                  0
occupation          28
agecat               1
dtype: int64


In [ ]:
## Step 1: Record Filtering & Identifier Type Casting

### Actions:
1. **Remove Corrupted Test Row**: Row index 0 (`inpatient_number = 5`) is a dummy test 
   record containing `NaN` across all attributes with an isolated BMI value.
2. **Convert ID Data Type**: Convert `inpatient_number` from numerical `int64` to `str` 
   to avoid misidentifying patient IDs as quantitative numbers in statistical summaries.

In [24]:
# 1. Drop dummy/corrupted row
df_clean = df_clean[df_clean["inpatient_number"] != 5].copy()

# 2. Convert ID to string data type
df_clean["inpatient_number"] = df_clean["inpatient_number"].astype(str)

# Verify change
print(f"Updated Shape: {df_clean.shape}")
print(f"inpatient_number Data Type: {df_clean['inpatient_number'].dtype}")

Updated Shape: (2008, 7)
inpatient_number Data Type: str


In [ ]:
## Step 2: Biological Outlier Detection & Calculation Fixes

### Actions:
1. **Flag Impossible Measurements**: 
   - Heights under 1.0 (0.35m - 0.60 m) are adult data entry errors and distort calculated BMIs (> 400).
   - Weights < 10.0 kg including 0.0 kg are invalid.
2. **Set Anomalies to `NaN`**: Replace invalid physical measurements with `NaN`.
3. **Recalculate BMI**: Recompute accurate BMIs using the standard formula:
   $$\text{BMI} = \frac{\text{Weight (kg)}}{\text{Height (m)}^2}$$

In [26]:
# 1. Create missingness flags for auditability
df_clean["flag_invalid_height"] = df_clean["height"] < 1.0
df_clean["flag_invalid_weight"] = df_clean["weight"] <= 10.0

# 2. Replace impossible entries with NaN
df_clean.loc[df_clean["flag_invalid_height"], "height"] = np.nan
df_clean.loc[df_clean["flag_invalid_weight"], "weight"] = np.nan

# 3. Recalculate true BMI
df_clean["bmi"] = df_clean["weight"] / (df_clean["height"] ** 2)

# Display summary statistics after cleanup
print("--- Sanitized BMI Summary Statistics ---")
print(df_clean["bmi"].describe())

--- Sanitized BMI Summary Statistics ---
count    2000.000000
mean       21.296969
std         3.807980
min        13.061224
25%        18.491124
50%        20.761246
75%        23.437500
max        39.111111
Name: bmi, dtype: float64


In [ ]:
## Step 3: Categorical Standardization & Feature Engineering

### Actions:
1. **Standardize Text Fields**: Fix inconsistent casing in `occupation` (e.g., `farmer` $\rightar
`Farmer`, `UrbanResident` $\rightarrow$ `Urban Resident`) and handle missing values by filling 
them with `"Unspecified"`.
2. **Ordered Categorical Conversion**: Convert `agecat` into an ordered categorical data type 
(`21-29` < `29-39` < ... < `89-110`) to preserve correct age progression in charts.
3. **Feature Engineering**: Bin continuous BMI values into World Health Organization clinical
risk categories (`Underweight`, `Normal`, `Overweight`, `Obese`).

In [27]:
# 1. Standardize occupation names
occupation_map = {
    "farmer": "Farmer",
    "worker": "Worker",
    "UrbanResident": "Urban Resident",
    "Officer": "Officer",
    "Others": "Others",
}
df_clean["occupation"] = (
    df_clean["occupation"].map(occupation_map).fillna("Unspecified")
)

# 2. Set Gender categorical
df_clean["gender"] = df_clean["gender"].astype("category")

# 3. Set Ordered Categorical Age Ranges
age_order = [
    "21-29",
    "29-39",
    "39-49",
    "49-59",
    "59-69",
    "69-79",
    "79-89",
    "89-110",
]
df_clean["agecat"] = pd.Categorical(
    df_clean["agecat"], categories=age_order, ordered=True
)

# 4. Feature Engineering: Clinical BMI Risk Categories
bmi_bins = [0, 18.5, 24.9, 29.9, np.inf]
bmi_labels = ["Underweight", "Normal", "Overweight", "Obese"]
df_clean["bmi_category"] = pd.cut(
    df_clean["bmi"], bins=bmi_bins, labels=bmi_labels
)

# Check value distributions
print("--- Occupation Distribution ---")
print(df_clean["occupation"].value_counts())
print("\n--- BMI Risk Category Distribution ---")
print(df_clean["bmi_category"].value_counts())

--- Occupation Distribution ---
occupation
Urban Resident    1670
Farmer             198
Others              89
Unspecified         27
Worker              17
Officer              7
Name: count, dtype: int64

--- BMI Risk Category Distribution ---
bmi_category
Normal         1163
Underweight     507
Overweight      263
Obese            67
Name: count, dtype: int64


In [ ]:
## Step 4: Final Validation & Export

### Actions:
1. Confirm final column data types and check remaining null counts.
2. Export the clean dataset to `Demography_Cleaned.csv` for downstream merging.

In [28]:
# 1. Final datatypes check
print("--- Final Data Types ---")
print(df_clean.dtypes)

# 2. Final missing value audit
print("\n--- Final Null Counts ---")
print(df_clean.isnull().sum())

# 3. Export cleaned data
df_clean.to_csv("Demography_Cleaned.csv", index=False)
print(
    f"\nDataset cleaned successfully! Output saved with shape {df_clean.shape}."
)

--- Final Data Types ---
inpatient_number            str
gender                 category
weight                  float64
height                  float64
bmi                     float64
occupation                  str
agecat                 category
flag_invalid_height        bool
flag_invalid_weight        bool
bmi_category           category
dtype: object

--- Final Null Counts ---
inpatient_number       0
gender                 0
weight                 4
height                 4
bmi                    8
occupation             0
agecat                 0
flag_invalid_height    0
flag_invalid_weight    0
bmi_category           8
dtype: int64

Dataset cleaned successfully! Output saved with shape (2008, 10).


In [ ]:
*Check or Open the File Path Using Code
Run this quick code cell in Jupyter Notebook to display the exact folder path where your file was saved:

In [29]:
import os

# Print the full directory path where your notebook and CSV are saved
print("File saved at:")
print(os.path.abspath("Demography_Cleaned.csv"))

File saved at:
C:\Users\apriy\Documents\GitHub\Team4_The_Zen_OF_Five_PythonHackathon_Sep2026\Demography_Cleaned.csv


In [ ]:
* Verify the Saved File


In [30]:
# Check if the cleaned file exists and inspect the first few rows
df_check = pd.read_csv("Demography_Cleaned.csv")
print(
    f"Successfully loaded saved file! Rows: {df_check.shape[0]}, Columns: {df_check.shape[1]}"
)
df_check.head()

Successfully loaded saved file! Rows: 2008, Columns: 10


,inpatient_number,gender,weight,height,bmi,occupation,agecat,flag_invalid_height,flag_invalid_weight,bmi_category
0,827040,Female,50.0,1.45,23.781213,Unspecified,69-79,False,False,Normal
1,857781,Male,50.0,1.64,18.590125,Urban Resident,69-79,False,False,Normal
2,743087,Female,51.0,1.63,19.195303,Urban Resident,69-79,False,False,Normal
3,866418,Male,70.0,1.70,24.221453,Farmer,59-69,False,False,Normal
4,775928,Male,65.0,1.70,22.491349,Urban Resident,69-79,False,False,Normal


In [ ]:
#Priya Demography table Data cleaning ends here

In [ ]:
#Priya Patient Presciption Data Cleaning starts here

In [ ]:
# Hackathon Data Cleaning & Reshaping Pipeline: Patient Prescriptions Table

## Executive Overview
This notebook processes the `patient_prescriptions.csv` transactional dataset. 

### Key Processing Actions:
1. **Load & Copy**: Load the raw transactional data and create an isolated working copy.
2. **Identifier & Text Cleaning**: Convert `inpatient_number` to string data type and strip unintended whitespace from drug names.
3. **Reshaping (Long to Wide Format)**:
   - Compute the total medication burden (`total_prescriptions_count`) per patient.
   - One-hot encode the **25 exact unique drug names** directly from the raw dataset rows into binary columns (`1` = Prescribed, `0` = Not Prescribed) so each patient has exactly one row.
4. **Validation & Export**: Verify structural integrity and export the final patient-level table.

In [37]:

import numpy as np
import pandas as pd

# 1. Load raw dataset (adjust folder path if your file is inside a subfolder like 'cardiac_failure/')
df_prescriptions_raw = pd.read_csv("cardiac_failure/patient_precriptions.csv")

# 2. Create a working copy to preserve raw data in memory
df_rx = df_prescriptions_raw.copy()

# 3. Baseline Audit
print(f"Raw Dataset Dimensions: {df_rx.shape[0]} rows, {df_rx.shape[1]} columns")
print(f"Unique Patients Count: {df_rx['inpatient_number'].nunique()}")
print(f"Unique Drug Names Count: {df_rx['drug_name'].nunique()}")
print("\n--- Initial Null Values Audit ---")
print(df_rx.isnull().sum())

Raw Dataset Dimensions: 15362 rows, 2 columns
Unique Patients Count: 2007
Unique Drug Names Count: 25

--- Initial Null Values Audit ---
inpatient_number    0
drug_name           0
dtype: int64


In [ ]:
## Step 1: Identifier Type Casting & Text Standardization

### Actions:
1. Convert `inpatient_number` from `int64` to `str` to ensure consistent keys when merging with Demography and Clinical tables.
2. Strip unexpected leading/trailing whitespaces from `drug_name` strings to prevent text formatting mismatches.

In [38]:
# 1. Convert patient ID to string data type
df_rx["inpatient_number"] = df_rx["inpatient_number"].astype(str)

# 2. Clean whitespace in drug names
df_rx["drug_name"] = df_rx["drug_name"].str.strip()

# Verify data type change
print(f"inpatient_number Data Type: {df_rx['inpatient_number'].dtype}")

inpatient_number Data Type: str


In [ ]:
## Step 2: Aggregation & Reshaping (Long Format to Wide Format)

### Actions:
Machine learning models require **one row per patient**. We transform the transactional rows using:
1. **Total Prescription Count**: Group by `inpatient_number` to calculate total prescriptions per patient (`total_prescriptions_count`).
2. **One-Hot Encoding (Pivot)**: Pivot the 25 exact drug names present in the raw data into binary indicator columns (`1` or `0`).
3. **Merge Features**: Combine count and individual drug flags into a unified 1-row-per-patient dataset (**27 columns total**).

--- Drug Count by Therapeutic Class ---
drug_class
class_diuretic                      5727
class_inotrope_glycoside            2742
class_other                         2052
class_antiplatelet_anticoagulant    1878
class_beta_blocker                   830
class_statin                         822
class_acei_arb                       782
class_vasodilator_nitrate            529
Name: count, dtype: int64


In [ ]:
## Step 3: Reshaping from Long Format to Patient-Level Wide Format

### Actions:
1. **Total Medication Count**: Calculate total number of drugs prescribed to each patient (`total_prescriptions_count`).
2. **One-Hot Encoding**: Pivot individual drug names into binary indicator columns (`1` = Prescribed, `0` = Not Prescribed).
3. **Class Indicator Encoding**: Pivot therapeutic drug classes into binary flags (`1` = On Class Therapy, `0` = Not On Class Therapy).

In [39]:
# 1. Calculate total prescription count per patient
df_patient_count = (
    df_rx.groupby("inpatient_number")["drug_name"]
    .count()
    .reset_index(name="total_prescriptions_count")
)

# 2. One-Hot Encode the exact 25 drug names directly from dataset rows
df_drugs_pivot = (
    pd.crosstab(df_rx["inpatient_number"], df_rx["drug_name"])
    .clip(upper=1)
    .reset_index()
)

# Clean drug column names for pythonic access (spaces/hyphens converted to underscores with 'rx_' prefix)
df_drugs_pivot.columns = [
    col if col == "inpatient_number" else f"rx_{col.lower().replace(' ', '_')}"
    for col in df_drugs_pivot.columns
]

# 3. Merge into a single Patient-Level Wide Table
df_prescriptions_patient_level = df_patient_count.merge(
    df_drugs_pivot, on="inpatient_number"
)

# Display transformed structure
print("Transformed Patient-Level Prescriptions Table Preview:")
print(f"Shape: {df_prescriptions_patient_level.shape} (Expected: 2007 rows, 27 columns)")
df_prescriptions_patient_level.head()

Transformed Patient-Level Prescriptions Table Preview:
Shape: (2007, 27) (Expected: 2007 rows, 27 columns)


,inpatient_number,total_prescriptions_count,rx_aspirin_enteric-coated_tablet,rx_atorvastatin_calcium_tablet,rx_benazepril_hydrochloride_tablet,rx_clopidogrel_hydrogen_sulphate_tablet,rx_deslanoside_injection,rx_digoxin_tablet,rx_dobutamine_hydrochloride_injection,rx_enoxaparin_sodium_injection,...,rx_metoprolol_succinate_sustained-release_tablet,rx_milrinone_injection,rx_nitroglycerin_injection,rx_shenfu_injection,rx_spironolactone_tablet,rx_torasemide_tablet,rx_valsartan_dispersible_tablet,rx_metoprolol_tartrate_injection,rx_sulfotanshinone_sodium_injection,rx_warfarin_sodium_tablet
0,722128,7,0,0,0,0,1,1,0,0,...,0,1,0,0,1,0,0,0,0,1
1,723327,12,1,1,0,1,1,1,0,0,...,0,1,0,0,1,1,1,0,0,0
2,723617,4,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,724385,6,0,0,0,0,1,1,0,0,...,0,1,0,0,1,0,0,0,0,0
4,725509,9,0,0,0,0,1,1,0,0,...,0,1,0,1,1,1,0,1,0,0


In [ ]:
## Step 3: Final Validation, Path Inspection & Export

### Actions:
1. Validate that the patient count equals 2,007 unique patients and check for missing values.
2. Export the clean dataset to `Patient_Prescriptions_Cleaned.csv` in your working directory.
3. Print the exact file save path to confirm location.

In [41]:
import os

# 1. Validation Checks
assert (
    df_prescriptions_patient_level["inpatient_number"].nunique() == 2007
), "Patient count mismatch!"
assert (
    df_prescriptions_patient_level.isnull().sum().sum() == 0
), "Null values found in transformed table!"

# 2. Export clean CSV
output_filename = "Patient_Prescriptions_Cleaned.csv"
df_prescriptions_patient_level.to_csv(output_filename, index=False)

# 3. Verification & Absolute File Path Display
print("--- Export Summary ---")
print(f"File Saved Successfully: {os.path.exists(output_filename)}")
print(f"Absolute File Path: {os.path.abspath(output_filename)}")
print(f"Final Output Shape: {df_prescriptions_patient_level.shape}")
print(f"Total Columns Count: {df_prescriptions_patient_level.shape[1]}")

--- Export Summary ---
File Saved Successfully: True
Absolute File Path: C:\Users\apriy\Documents\GitHub\Team4_The_Zen_OF_Five_PythonHackathon_Sep2026\Patient_Prescriptions_Cleaned.csv
Final Output Shape: (2007, 27)
Total Columns Count: 27


In [ ]:
# Priya Patient_prescription Data cleaning ends here